In [520]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error


# 1

In [498]:
geo_data_0 = pd.read_csv("../datasets/geo_data_0.csv")
geo_data_1 = pd.read_csv("../datasets/geo_data_1.csv")
geo_data_2 = pd.read_csv("../datasets/geo_data_2.csv")


# 2

In [499]:
def features_target_split(data_df:pd.DataFrame,non_training_columns:list[str],target_column:str) -> tuple[pd.DataFrame,pd.Series]:
    features = data_df.drop(non_training_columns + [target_column],axis=1)
    target = data_df.loc[:,target_column]
    return features, target

In [500]:
def model_pipe(num_columns:list[str]):
    scaler = StandardScaler()
    ct = ColumnTransformer(
        transformers=[("num_preprocess", scaler, num_columns)],
        remainder="passthrough"
    )

    pipe = Pipeline([
        ("column_transform", ct),
        ("model", LinearRegression())
    ])
    return pipe

In [521]:
def region_workflow(data_df:pd.DataFrame,non_training_columns,feature_columns,target_column,test_size:float,random_state:int) -> tuple[pd.Series,pd.Series,pd.DataFrame,float,float,Pipeline]:

    features, target = features_target_split(data_df, non_training_columns,target_column)
    features_train, features_valid, target_train, target_valid = train_test_split(features, target, test_size=test_size,random_state=random_state)

    pipe = model_pipe(feature_columns)
    pipe.fit(features_train,target_train)

    predicted_valid = pd.Series(pipe.predict(features_valid),index=target_valid.index)
    validation_results = pd.DataFrame({"actual_product":target_valid,"predicted_product":predicted_valid}).merge(data_df[non_training_columns],how="left",left_index=True,right_index=True)

    mean_predicted = predicted_valid.mean()
    root_mean_square_error = mean_squared_error(target_valid,predicted_valid) ** (1/2)

    return target_valid, predicted_valid, validation_results, mean_predicted,root_mean_square_error,pipe

In [503]:
split_state = 12345
train_size = 0.75
valid_size = 0.25
non_training_columns = ["id"]
feature_columns = ["f0","f1","f2"]
target_column = "product"


In [504]:
target_valid_0, predicted_valid_0, validation_results_0, mean_predicted_0, rmse_0, pipe_0 = region_workflow(geo_data_0,non_training_columns,feature_columns,target_column,valid_size,split_state)
print(f"Volume médio previsto de reservas: {mean_predicted_0:.2f}")
print(f"REQM do modelo: {rmse_0:.2f}")


Volume médio previsto de reservas: 92.59
REQM do modelo: 37.58


In [505]:
target_valid_1, predicted_valid_1, validation_results_1, mean_predicted_1, rmse_1, pipe_1 = region_workflow(geo_data_1,non_training_columns,feature_columns,target_column,valid_size,split_state)
print(f"Volume médio previsto de reservas: {mean_predicted_1:.2f}")
print(f"REQM do modelo: {rmse_1:.2f}")


Volume médio previsto de reservas: 68.73
REQM do modelo: 0.89


In [506]:
target_valid_2, predicted_valid_2, validation_results_2, mean_predicted_2, rmse_2, pipe_2 = region_workflow(geo_data_2,non_training_columns,feature_columns,target_column,valid_size,split_state)
print(f"Volume médio previsto de reservas: {mean_predicted_2:.2f}")
print(f"REQM do modelo: {rmse_2:.2f}")


Volume médio previsto de reservas: 94.97
REQM do modelo: 40.03


# Análise

- Região 0 tem volume médio previsto de reservas de 92.59 milhares de barris de petróleo com REQM 37.58, o que indica que as predições são incertas.

- Região 1 tem volume médio previsto de reservas de 68.73 milhares de barris de petróleo com REQM 0.89, o que indica que as predições tem alto nível de confiabilidade.

- Região 2 tem volume médio previsto de reservas de 94.97 milhares de barris de petróleo com REQM 40.03, o que indica que as predições são incertas.

Nenhuma das três regiões pode ser escolhida como a melhor região para desenvolvimento usando somente essas informações. Mas podemos afirmar que, aplicando a metodologia de treinamento desses modelos, apenas o modelo da região 1 parece ser confiável para futuros poços (assumindo que as características região não mudem e que os dados usados para treinamento são representativos da região).

# 3.

In [525]:
# Número de poços de Petróleo para desenvolvimento
oil_well_num = 200

# Orçamento de 1 milhões de dólares ($1.000.000) para o desenvolvimento
budget = 100_000_000

# Receita de uma unidade de produto (que equivalem a 1.000 barris de petróleo).
product_unit_rev = 4500

# Risco de perdas máximo em porcentagem. Manter apenas as regiões com risco inferior.
loss_risk_max = 0.025


In [524]:
# Dado o investimento de 100 milhões para 200 poços de petróleo, cada um precisa
# produzir, em média, uma quantidade de unidades equivalente a pelo menos 500 mil
# dólares para evitar prejuízos (isso é aproximadamente 111,1 unidades).
product_mean_min = budget / oil_well_num / product_unit_rev
print(product_mean_min)

111.11111111111111


In [508]:
print(f"Volume médio mínimo de barris para evitar prejuízos: {product_mean_min:.2f}")
product_mean_0 =geo_data_0["product"].mean()
product_mean_1 =geo_data_1["product"].mean()
product_mean_2 =geo_data_2["product"].mean()
print(f"Volume médio de barris da Região 0: {product_mean_0:.2f}")
print(f"Volume médio de barris da Região 1: {product_mean_1:.2f}")
print(f"Volume médio de barris da Região 2: {product_mean_2:.2f}")

Volume médio mínimo de barris para evitar prejuízos: 111.11
Volume médio de barris da Região 0: 92.50
Volume médio de barris da Região 1: 68.83
Volume médio de barris da Região 2: 95.00


Nenhuma das três regiões apresenta um volume médio de barris de petróleo que supere o volume médio mínimo para evitarmos prejuízos. Isso não significa que nenhuma região gere lucros, mas significa que termos que ser seletivos ao escolher os pontos para possível desenvolvimento de poços de petróleo, uma vez que nenhum região é, __em média__, ideal.

# 4.

In [509]:
best_wells_0 = validation_results_0.sort_values(by="predicted_product",ascending=False).head(200)
best_wells_1 = validation_results_1.sort_values(by="predicted_product",ascending=False).head(200)
best_wells_2 = validation_results_2.sort_values(by="predicted_product",ascending=False).head(200)


In [526]:
def region_predicted_revenue_profit(region_data:pd.DataFrame,sort_column:str,target_column:str,count:int, budget: int) -> tuple[pd.DataFrame,float]:
    top_wells = region_data.sort_values(by=sort_column,ascending=False).head(count)
    target_product_sum = top_wells[target_column].sum()
    potential_revenue = target_product_sum * product_unit_rev
    profit = potential_revenue - budget
    return top_wells,potential_revenue, profit



In [535]:
top_wells_0, potential_revenue_0, profit_0 = region_predicted_revenue_profit(
    validation_results_0, "predicted_product","actual_product", oil_well_num, budget
)
top_wells_1, potential_revenue_1, profit_1 = region_predicted_revenue_profit(
    validation_results_1, "predicted_product","actual_product", oil_well_num, budget
)
top_wells_2, potential_revenue_2, profit_2 = region_predicted_revenue_profit(
    validation_results_2, "predicted_product","actual_product", oil_well_num, budget
)

print(f"Receita potencial da Região 0 em dólares: {potential_revenue_0:,.2f}")
print(f"Receita potencial da Região 1 em dólares: {potential_revenue_1:,.2f}")
print(f"Receita potencial da Região 2 em dólares: {potential_revenue_2:,.2f}")
print()
print(f"Lucro potencial da Região 0 em dólares: {profit_0:,.2f}")
print(f"Lucro  potencial da Região 1 em dólares: {profit_1:,.2f}")
print(f"Lucro potencial da Região 2 em dólares: {profit_2:,.2f}")

Receita potencial da Região 0 em dólares: 133,208,260.43
Receita potencial da Região 1 em dólares: 124,150,866.97
Receita potencial da Região 2 em dólares: 127,103,499.64

Lucro potencial da Região 0 em dólares: 33,208,260.43
Lucro  potencial da Região 1 em dólares: 24,150,866.97
Lucro potencial da Região 2 em dólares: 27,103,499.64


A Região 0 é a região com o maior valor de lucro potencial se escolhermos exatamente os pontos que foram escolhidos pelo nosso modelo. Isso significa que, se esses pontos forem escolhidos para construção, a Região 0 é a melhor região para construção de 200 poços de petróleo, com um potencial de lucro de 33.2 milhões de dólares.

5.

In [512]:
# Intervalo de Confiança de 95% para a Receita Média (2.5% para cada cauda da
# distribuição)
confidence_interval = (0.025,0.975)

bootstrap_sample_size = 500
bootstrap_samples = 1000

In [513]:
def profit_distribution(region_data:pd.DataFrame,sort_column:str,target_column:str,bootstrap_samples:int,bootstrap_sample_size:int,top_well_num:int, confidence_interval:tuple[float,float]):

    bootstrap_state = np.random.RandomState(12345)
    profits = []

    for i in range(bootstrap_samples):
        subsample = region_data.sample(n=bootstrap_sample_size, replace=True,random_state=bootstrap_state)
        sorted_subsample = subsample.sort_values(by=sort_column,ascending=False)
        top_subsample = sorted_subsample.head(top_well_num)
        total_revenue = top_subsample[target_column].sum() * product_unit_rev
        profit = total_revenue - budget
        profits.append(profit)


    profits = pd.Series(profits)
    lower = profits.quantile(confidence_interval[0])
    upper = profits.quantile(confidence_interval[1])
    profit_mean = profits.mean()
    loss_proba = (profits <0).mean()

    return profit_mean, lower, upper, loss_proba, profits


In [517]:
profit_mean_0, lower_0, upper_0, loss_proba_0, profits_0 = profit_distribution(validation_results_0,"predicted_product", "actual_product", bootstrap_samples, bootstrap_sample_size, oil_well_num, confidence_interval)

print(f"Média de Lucro para Região 0 (usando técnica bootstrapping): {profit_mean_0:,.2f}")
print(f"Quantil 2.5%: {lower_0:,.2f}")
print(f"Quantil 97.5%: {upper_0:,.2f}")
print(f"Probabilidade de Prejuízos: {loss_proba_0:,.2%}")



Média de Lucro para Região 0 (usando técnica bootstrapping): 3,961,649.85
Quantil 2.5%: -1,112,155.46
Quantil 97.5%: 9,097,669.42
Probabilidade de Prejuízos: 6.90%


In [536]:
profit_mean_1, lower_1, upper_1, loss_proba_1, profits_1 = profit_distribution(validation_results_1,"predicted_product", "actual_product", bootstrap_samples, bootstrap_sample_size, oil_well_num, confidence_interval)

print(f"Média de Lucro para Região 1 (usando técnica bootstrapping): {profit_mean_1:,.2f}")
print(f"Quantil 2.5%: {lower_1:,.2f}")
print(f"Quantil 97.5%: {upper_1:,.2f}")
print(f"Probabilidade de Prejuízos: {loss_proba_1:,.2%}")


Média de Lucro para Região 1 (usando técnica bootstrapping): 4,560,451.06
Quantil 2.5%: 338,205.09
Quantil 97.5%: 8,522,894.54
Probabilidade de Prejuízos: 1.50%


In [537]:
profit_mean_2, lower_2, upper_2, loss_proba_2, profits_2 = profit_distribution(validation_results_2,"predicted_product", "actual_product", bootstrap_samples, bootstrap_sample_size, oil_well_num, confidence_interval)

print(f"Média de Lucro para Região 2 (usando técnica bootstrapping): {profit_mean_2:,.2f}")
print(f"Quantil 2.5%: {lower_2:,.2f}")
print(f"Quantil 97.5%: {upper_2:,.2f}")
print(f"Probabilidade de Prejuízos: {loss_proba_2:,.2%}")


Média de Lucro para Região 2 (usando técnica bootstrapping): 4,044,038.67
Quantil 2.5%: -1,633,504.13
Quantil 97.5%: 9,503,595.75
Probabilidade de Prejuízos: 7.60%


Conclusão